# Post-processing and standard plots

This tutorial introduces the standardized post-processing interface. We run a small Vlasov–Ampère example, get its output as an autocomplete-friendly `Output`, and make the plots most commonly used to inspect a simulation.

For a production run you can skip the simulation setup and open its output folder with `struphy.open_output("path/to/sim")` instead.

In [ ]:
import os
import tempfile

from struphy import (
    BinningPlot,
    BoundaryParameters,
    DerhamOptions,
    EnvironmentOptions,
    LoadingParameters,
    SavingParameters,
    Simulation,
    SortingParameters,
    Time,
    WeightsParameters,
    domains,
    grids,
    maxwellians,
    perturbations,
)
from struphy.models import VlasovAmpereOneSpecies

## Create a compact demonstration run

Post-processing operates on a completed run. The small setup below saves an electric field, a few marker trajectories, scalar diagnostics, and a binned $(\eta_1,v_1)$ distribution. These are the main output types handled by the plotting interface.

In [ ]:
model = VlasovAmpereOneSpecies(alpha=1.0, epsilon=-1.0, with_B0=False)
model.em_fields.e_field.save_data = True
model.em_fields.phi.save_data = True
model.kinetic_ions.var.save_data = True

model.propagators.push_eta.options = model.propagators.push_eta.Options()
model.propagators.coupling_va.options = model.propagators.coupling_va.Options()
model.initial_poisson.options = model.initial_poisson.Options(stab_mat="M0")

binplot = BinningPlot(
    slice="e1_v1",
    n_bins=(32, 32),
    ranges=((0.0, 1.0), (-5.0, 5.0)),
)
model.kinetic_ions.set_markers(
    loading_params=LoadingParameters(ppc=32, seed=1234),
    weights_params=WeightsParameters(control_variate=True),
    boundary_params=BoundaryParameters(),
    sorting_params=SortingParameters(boxes_per_dim=(4, 1, 1), do_sort=True),
    saving_params=SavingParameters(n_markers=12, binning_plots=(binplot,)),
)

background = maxwellians.Maxwellian3D(n=(1.0, None))
model.kinetic_ions.var.add_background(background)
density_mode = perturbations.ModesCos(ls=(1,), amps=(1e-3,))
model.kinetic_ions.var.add_initial_condition(maxwellians.Maxwellian3D(n=(1.0, density_mode)))

In [ ]:
demo_tmp = tempfile.TemporaryDirectory(prefix="struphy_postprocessing_")
demo_root = demo_tmp.name

env = EnvironmentOptions(
    out_folders=demo_root,
    sim_folder="vlasov_ampere_demo",
    save_restart=False,
)
sim = Simulation(
    model=model,
    env=env,
    time_opts=Time(dt=0.1, Tend=0.4),
    domain=domains.Cuboid(r1=2 * 3.141592653589793),
    grid=grids.TensorProductGrid(num_elements=(8, 1, 1)),
    derham_opts=DerhamOptions(degree=(2, 1, 1)),
)
out = sim.run()
print(f"Raw output: {sim.env.path_out}")

## Process and load the output

`sim.run()` returns the run's output as a `Output`, which is also available later as `sim.output`. Scalars are read directly from the raw output; fields and particle products need post-processing, which runs with default options the first time they are accessed.

To choose options, call `out.process()` first. It evaluates saved FEEC fields and organizes particle diagnostics; `physical=True` additionally creates physical field components. Existing products made with the same options are reused, so re-running a cell is cheap.

Individual products are standard `xarray.DataArray` objects with named dimensions, coordinates, units, and labels. Arrays are loaded only when accessed. The simulation that produced them is `out.sim`.

In [ ]:
out.process(physical=True)

Products are arranged into clear namespaces. VS Code and interactive shells can complete the available names after a run is opened: fields are grouped by field species, while distribution and density products are grouped by species and saved slice. Every product can also be looked up by name, e.g. `out["electric_energy"]` or `out["kinetic_ions/e1_v1_density/f_binned"]`; flat catalogs remain available for code that needs to iterate over arbitrary products.

In [ ]:
print("scalars:", tuple(out.scalars.data_vars))
print("field species:", tuple(out.fields))
print("distribution species:", tuple(out.distributions))
print("particle species:", tuple(out.orbits))
print("all field products:", tuple(out.field_catalog))

phase_space = out.distributions.kinetic_ions.e1_v1_density.f_binned
print(phase_space)
print("dimensions:", phase_space.dims)
print("time coordinate:", phase_space.t)

## Scalar overview and time series

All standard plots are methods of `out.plot`, so no further imports are needed. They accept a product name or any array, and titles carry the run's numerical parameters.

`out.plot.scalars()` gives a quick overview of every recorded scalar, with the relative error of the total energy below. `out.plot.timeseries()` shows individual series on linear or logarithmic axes; `fit=(t0, t1)` adds an exponential fit restricted to that time window. Plots return an already-rendered `PlotResult`, which a notebook displays by itself; calling `.save()` never draws a second figure.

In [ ]:
scalar_plot = out.plot.scalars()
energy_error = scalar_plot.data["relative_error"]
scalar_plot

In [ ]:
t_fit = 0.4 * out.sim.model.units.t  # in seconds, like every time coordinate of this run
energy_plot = out.plot.timeseries(
    "electric_energy",
    fit=(0.0, t_fit),
    fit_amplitude=True,
    title="Electric-field energy",
)
print("growth rate:", energy_plot.fit_results[0].rate)

# the same fit without a figure
print("growth rate:", out.analysis.growth_rate("electric_energy", window=(0.0, t_fit), amplitude=True).rate)

## Two-dimensional data

Choose the displayed dimensions with `x` and `y`, and fix all others with `isel` (by index) or `select` (by nearest coordinate value). Arrays can also be sliced beforehand with xarray's `.isel()` and `.sel()`. `coords="physical"` draws on the mapped coordinates instead of the logical ones.

In [ ]:
out.plot.slice(
    phase_space,
    x="e1",
    y="v1",
    isel={"t": -1},
    equal_aspect=False,
    title="Final phase-space distribution",
)

For a compact view of the evolution, `out.plot.panels()` chooses evenly spaced snapshots in time. `shared_clim=True` makes panel colors directly comparable.

In [ ]:
out.plot.panels(
    phase_space,
    x="e1",
    y="v1",
    nrows=1,
    ncols=5,
    title="Phase-space evolution",
)

## Interactive plots

`out.plot.viewer()` adds one slider for every dimension not assigned to the display axes. In JupyterLab, run `%matplotlib widget` before this cell if `ipympl` is installed; the default inline backend still displays the initial frame. Keep the viewer alive so its callbacks remain connected. `out.plot.animation()` and `out.plot.frames()` sweep the same way.

In [ ]:
phase_viewer = out.plot.viewer(phase_space, x="e1", y="v1")
phase_viewer

Saved marker orbits are grouped by species. `out.plot.orbits()` draws their three-dimensional paths, while `max_markers` limits rendering cost for large production runs.

In [ ]:
out.plot.orbits("kinetic_ions", max_markers=12, show_paths=True)

## Save standard output

Every `PlotResult` supports `.save(path)`. For a complete scalar report, `out.save_report()` writes a CSV table, an overview, and one PNG per scalar beneath `post_processing/report/`.

In [ ]:
written = out.save_report()
print("Wrote:")
for path in written:
    print(" ", os.path.relpath(path, out.path_out))

## Apply the workflow to another run

For an already completed simulation, possibly in a separate process without MPI, open its output folder:

```python
import struphy

out = struphy.open_output("/path/to/sim_1").process(physical=True)
out.sim.domain, out.sim.model.units  # the simulation, restored from disk without allocating
```

Use `out.scalars`, `out.fields`, `out.distributions`, `out.orbits`, and `out.densities`. Attribute access is the normal interactive API; the corresponding `*_catalog` mappings are intended for generic loops and tooling.